# Create Versus Arthritis Awards

**Versus Arthritis** (funder_id `4320327444`, IAMHRF expanded, priority `313`).
arthritis-uk.org live research-projects directory (61 projects; old domain 403s).
100% PI/institution/title/GBP amount/start-date/scheme; native Reference ids on
50/61 (11 publish literal "TBC" → slug fallback ids).

Parquet: `s3://openalex-ingest/awards/versus_arthritis/versus_arthritis_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.versus_arthritis_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/versus_arthritis/versus_arthritis_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.versus_arthritis_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.versus_arthritis_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.versus_arthritis_raw LIMIT 5;

## Step 2: Create Versus Arthritis Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.versus_arthritis_awards
USING delta
AS
WITH
va_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320327444  -- Versus Arthritis
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        CAST(NULL AS STRING) as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'GBP' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'versus_arthritis' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'd MMMM yyyy') as start_date,
        CAST(NULL AS DATE) as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'd MMMM yyyy')) as start_year,
        CAST(NULL AS INT) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'United Kingdom' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.versus_arthritis_raw g
    CROSS JOIN va_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'versus_arthritis' AND priority = 313;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    313 as priority
FROM openalex.awards.versus_arthritis_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards FROM openalex.awards.versus_arthritis_awards;

In [ ]:
%sql
SELECT funder.display_name, COUNT(*) as cnt FROM openalex.awards.versus_arthritis_awards GROUP BY 1;

In [ ]:
%sql
SELECT
    COUNT(*) as total, COUNT(display_name) as has_title, COUNT(amount) as has_amount,
    COUNT(lead_investigator) as has_pi, COUNT(start_date) as has_start_date,
    ROUND(SUM(amount)/1000000, 2) as total_funding_million_gbp
FROM openalex.awards.versus_arthritis_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'versus_arthritis' AND priority = 313;